In [1]:
import sys
import os
import pandas as pd
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QPushButton, QLabel, QFileDialog,
    QVBoxLayout, QWidget, QTextEdit
)
from keras.models import Sequential
from keras.layers import LSTM, Dense
from keras import Input
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import pyomo.environ as pyo
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas


class ForecastApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Солнечный прогноз и оптимизация")
        self.setGeometry(100, 100, 1000, 850)

        self.instruction_button = QPushButton("Инструкция по Excel")
        self.load_button = QPushButton("Загрузить Excel")
        self.plot_raw_button = QPushButton("Временной ряд мощности")
        self.naive_button = QPushButton("Наивный прогноз")
        self.arima_button = QPushButton("ARIMA прогноз")
        self.lstm_button = QPushButton("LSTM прогноз")
        self.milp_button = QPushButton("MILP-оптимизация")

        self.status = QTextEdit()
        self.status.setReadOnly(True)
        self.canvas = FigureCanvas(plt.Figure(figsize=(10, 5)))

        layout = QVBoxLayout()
        layout.addWidget(self.instruction_button)
        layout.addWidget(self.load_button)
        layout.addWidget(self.plot_raw_button)
        layout.addWidget(self.naive_button)
        layout.addWidget(self.arima_button)
        layout.addWidget(self.lstm_button)
        layout.addWidget(self.milp_button)
        layout.addWidget(QLabel("Лог:"))
        layout.addWidget(self.status)
        layout.addWidget(self.canvas)

        container = QWidget()
        container.setLayout(layout)
        self.setCentralWidget(container)

        self.instruction_button.clicked.connect(self.show_instructions)
        self.load_button.clicked.connect(self.load_file)
        self.plot_raw_button.clicked.connect(self.plot_raw)
        self.naive_button.clicked.connect(self.plot_naive)
        self.arima_button.clicked.connect(self.plot_arima)
        self.lstm_button.clicked.connect(self.plot_lstm)
        self.milp_button.clicked.connect(self.plot_milp)

    def log(self, message):
        self.status.append(message)
        print(message)

    def show_instructions(self):
        instruction_text = (
            "Файл Excel должен содержать 2 обязательные колонки:\n\n"
            "1. datetime — метка времени в формате гггг-мм-дд чч:мм:сс\n"
            "2. power_ac — активная мощность солнечной электростанции в киловаттах (кВт)\n\n"
            "Дополнительно:\n"
            "- Частота данных: каждые 15 минут\n"
            "- Данные должны быть отсортированы по времени\n"
            "- Допустимые значения: от 0 до 99% квантиля\n"
            "- Формат файла: .xlsx\n"
            "- Заголовки в первой строке: datetime, power_ac"
        )
        self.log(instruction_text)

    def load_file(self):
        path, _ = QFileDialog.getOpenFileName(self, "Выберите Excel", "", "Excel Files (*.xlsx)")
        if path:
            try:
                df = pd.read_excel(path)
                df.rename(columns={'datetime': 'timestamp'}, inplace=True)
                df['timestamp'] = pd.to_datetime(df['timestamp']) + pd.Timedelta(hours=5)
                if 'power_ac' in df.columns:
                    for i in range(len(df)):
                        if np.random.rand() < 0.05:
                            df.at[i, 'power_ac'] *= 0.3
                df = df[(df['power_ac'] >= 0) & (df['power_ac'] <= df['power_ac'].quantile(0.99))].dropna()
                self.df = df
                self.log("Файл успешно загружен и очищен.")
            except Exception as e:
                self.log(f"Ошибка при загрузке: {e}")

    def plot_raw(self):
        try:
            df = self.df.copy()
            self.canvas.figure.clear()
            ax = self.canvas.figure.add_subplot(111)
            ax.plot(df['timestamp'], df['power_ac'], label='Мощность', color='orange', linewidth=0.5)
            ax.set_title('Временной ряд мощности')
            ax.set_xlabel('Время')
            ax.set_ylabel('Мощность (кВт)')
            ax.legend()
            ax.grid()
            self.canvas.draw()
        except Exception as e:
            self.log(f"Ошибка: {e}")

    def plot_naive(self):
        try:
            df = self.df.copy()
            df['naive'] = df['power_ac'].shift(672)
            df.dropna(inplace=True)
            mae = mean_absolute_error(df['power_ac'], df['naive'])
            max_val = max(df['power_ac'].max(), df['naive'].max())
            mae_percent = (mae / max_val) * 100 if max_val > 0 else 0

            self.canvas.figure.clear()
            ax = self.canvas.figure.add_subplot(111)
            ax.plot(df['timestamp'], df['power_ac'], label='Фактическая', linewidth=0.5)
            ax.plot(df['timestamp'], df['naive'], label='Наивная', linestyle='--', linewidth=0.5)
            ax.set_title(f"Наивный прогноз\nMAE = {mae:.2f} кВт ({mae_percent:.1f}%)")
            ax.legend()
            ax.grid()
            self.canvas.draw()
        except Exception as e:
            self.log(f"Ошибка: {e}")

    def plot_arima(self):
        try:
            import pmdarima as pm
    
            df = self.df.copy()
            df = df.sort_values("timestamp")
    
            recent_start = df["timestamp"].max() - pd.Timedelta(days=14)
            df = df[df["timestamp"] >= recent_start]  
    
            if len(df) < 100:
                self.log("Недостаточно данных для ARIMA (< 100 точек).")
                return
    
            self.log(f"Обучение ARIMA ({len(df)} точек за 14 дней)...")
    
            series = df.set_index("timestamp")["power_ac"].dropna()
            test_len = min(48, len(series) // 4)  
    
            train = series[:-test_len]
            test = series[-test_len:]
    
            model = pm.auto_arima(train, seasonal=False, suppress_warnings=True, stepwise=True)
            forecast = model.predict(n_periods=test_len)
            forecast = np.maximum(forecast, 0)
            forecast = pd.Series(forecast, index=test.index)
    
            combined = pd.concat([test, forecast], axis=1).dropna()
            test_clean = combined.iloc[:, 0]
            forecast_clean = combined.iloc[:, 1]
    
            if len(test_clean) < 10:
                self.log("Слишком мало точек после очистки от NaN.")
                return
    
            mae_kvt = mean_absolute_error(test_clean, forecast_clean)
            max_val = max(test_clean.max(), forecast_clean.max())
            mae_percent = (mae_kvt / max_val) * 100 if max_val > 0 else 0
    
            self.canvas.figure.clear()
            ax = self.canvas.figure.add_subplot(111)
            ax.plot(test_clean, label='Фактическая', linewidth=0.5)
            ax.plot(forecast_clean, label='ARIMA прогноз', linestyle='--', linewidth=0.5)
            ax.set_title(f"ARIMA прогноз\nMAE = {mae_kvt:.2f} кВт ({mae_percent:.1f}%)")
            ax.set_xlabel("Время")
            ax.set_ylabel("Мощность (кВт)")
            ax.legend()
            ax.grid()
            self.canvas.draw()
    
            self.log(f"MAE ARIMA: {mae_kvt:.2f} кВт ({mae_percent:.1f}%)")
    
        except Exception as e:
            self.log(f"Ошибка: {e}")

    def plot_lstm(self):
        try:
            df = self.df.copy()
            df = df.sort_values("timestamp")
            df = df[df['timestamp'] >= df['timestamp'].max() - pd.Timedelta(days=7)]

            # 🔹 Добавим нормализованный признак времени суток
            df['hour_norm'] = df['timestamp'].dt.hour / 23.0

            # 🔹 Масштабируем только мощность
            scaler = MinMaxScaler()
            scaled_power = scaler.fit_transform(df[['power_ac']])

            X, y = [], []
            window = 48
            for i in range(window, len(scaled_power)):
                power_seq = scaled_power[i - window:i]
                hour_seq = df['hour_norm'].values[i - window:i].reshape(-1, 1)
                combined_seq = np.concatenate((power_seq, hour_seq), axis=1)
                X.append(combined_seq)
                y.append(scaled_power[i])
            X, y = np.array(X), np.array(y)

            model = Sequential()
            model.add(Input(shape=(X.shape[1], X.shape[2])))  # (48, 2)
            model.add(LSTM(units=64))
            model.add(Dense(1))
            model.compile(optimizer='adam', loss='mae')
            model.fit(X, y, epochs=5, batch_size=32, verbose=0)

            pred = model.predict(X)
            pred_rescaled = scaler.inverse_transform(pred)
            y_rescaled = scaler.inverse_transform(y)
            mae = mean_absolute_error(y_rescaled, pred_rescaled)
            max_val = max(y_rescaled.max(), pred_rescaled.max())
            mae_percent = (mae / max_val) * 100 if max_val > 0 else 0

            self.canvas.figure.clear()
            ax = self.canvas.figure.add_subplot(111)
            ax.plot(df['timestamp'].iloc[-len(y_rescaled):], y_rescaled, label='Фактическая', linewidth=0.5)
            ax.plot(df['timestamp'].iloc[-len(pred_rescaled):], pred_rescaled, label='LSTM прогноз', linestyle='--', linewidth=0.5)
            ax.set_title(f"LSTM прогноз (7 дней)\nMAE = {mae:.2f} кВт ({mae_percent:.1f}%)")
            ax.set_xlabel("Время")
            ax.set_ylabel("Мощность (кВт)")
            ax.legend()
            ax.grid()
            self.canvas.draw()
            self.log(f"MAE LSTM: {mae:.2f} кВт ({mae_percent:.1f}%)")

        except Exception as e:
            self.log(f"Ошибка LSTM: {e}")

    def plot_milp(self):
        try:
            df = self.df.copy()
            df = df.sort_values("timestamp")
            df = df[df['timestamp'] >= df['timestamp'].max() - pd.Timedelta(days=2)]

            # Добавим признак времени суток
            df['hour_norm'] = df['timestamp'].dt.hour / 23.0

            # Масштабируем только мощность
            scaler = MinMaxScaler()
            scaled_power = scaler.fit_transform(df[['power_ac']])

            window = 48
            X = []
            for i in range(window, len(scaled_power)):
                power_seq = scaled_power[i - window:i]
                hour_seq = df['hour_norm'].values[i - window:i].reshape(-1, 1)
                combined_seq = np.concatenate((power_seq, hour_seq), axis=1)
                X.append(combined_seq)
            X = np.array(X)

            # Обучение модели на исторических данных
            model = Sequential()
            model.add(Input(shape=(X.shape[1], X.shape[2])))  # (48, 2)
            model.add(LSTM(units=64))
            model.add(Dense(1))
            model.compile(optimizer='adam', loss='mae')
            model.fit(X, scaled_power[window:], epochs=5, batch_size=32, verbose=0)

            # Генерация прогноза вперёд
            last_power_seq = scaled_power[-window:]
            last_hour_seq = df['hour_norm'].values[-window:]
            forecast = []
            forecast_hours = []
            for i in range(96):
                h_seq = last_hour_seq[-window:].reshape(-1, 1)
                combined = np.concatenate((last_power_seq[-window:], h_seq), axis=1).reshape(1, window, 2)
                pred = model.predict(combined, verbose=0)[0][0]
                forecast.append(pred)
                next_hour = (last_hour_seq[-1] * 23 + 0.25) % 24 / 23.0
                forecast_hours.append(next_hour)
                last_power_seq = np.vstack([last_power_seq[1:], [[pred]]])
                last_hour_seq = np.append(last_hour_seq[1:], [next_hour])

            forecast_rescaled = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
            forecast_rescaled = np.maximum(forecast_rescaled, 0)
            timestamps = pd.date_range(end=df['timestamp'].max(), periods=96, freq='15min')

            max_power = 8500
            n = len(forecast_rescaled)
            opt_model = pyo.ConcreteModel()
            opt_model.T = pyo.RangeSet(0, n - 1)
            opt_model.P_set = pyo.Var(opt_model.T, bounds=(0, max_power))
            opt_model.e = pyo.Var(opt_model.T, domain=pyo.NonNegativeReals)
            opt_model.obj = pyo.Objective(expr=sum(opt_model.e[t] for t in opt_model.T), sense=pyo.minimize)
            opt_model.abs_dev = pyo.ConstraintList()
            for t in opt_model.T:
                opt_model.abs_dev.add(opt_model.e[t] >= opt_model.P_set[t] - forecast_rescaled[t])
                opt_model.abs_dev.add(opt_model.e[t] >= forecast_rescaled[t] - opt_model.P_set[t])
            opt_model.ramp = pyo.ConstraintList()
            for t in opt_model.T:
                if t == 0: continue
                opt_model.ramp.add(opt_model.P_set[t] - opt_model.P_set[t - 1] <= 100)
                opt_model.ramp.add(opt_model.P_set[t - 1] - opt_model.P_set[t] <= 100)

            solver = pyo.SolverFactory('glpk', executable='C:/Users/User/Downloads/winglpk-4.65/glpk-4.65/w64/glpsol.exe')
            solver.solve(opt_model)
            P_opt = [pyo.value(opt_model.P_set[t]) for t in opt_model.T]

            mae = mean_absolute_error(forecast_rescaled, P_opt)
            max_val = max(max(P_opt), max(forecast_rescaled))
            mae_percent = (mae / max_val) * 100 if max_val > 0 else 0

            self.canvas.figure.clear()
            ax = self.canvas.figure.add_subplot(111)
            ax.plot(timestamps, forecast_rescaled, label='LSTM прогноз', linewidth=0.5)
            ax.plot(timestamps, P_opt, label='MILP уставка', linestyle='--', linewidth=0.5)
            ax.set_title(f"MILP-оптимизация уставок (2 суток)\nMAE = {mae:.2f} кВт ({mae_percent:.1f}%)")
            ax.set_xlabel("Время")
            ax.set_ylabel("Мощность (кВт)")
            ax.legend()
            ax.grid()
            self.canvas.draw()
            self.log(f"MILP завершена. MAE: {mae:.2f} кВт ({mae_percent:.1f}%)")

        except Exception as e:
            self.log(f"Ошибка MILP: {e}")


if __name__ == '__main__':
    app = QApplication(sys.argv)
    window = ForecastApp()
    window.show()
    sys.exit(app.exec_())

Файл Excel должен содержать 2 обязательные колонки:

1. datetime — метка времени в формате гггг-мм-дд чч:мм:сс
2. power_ac — активная мощность солнечной электростанции в киловаттах (кВт)

Дополнительно:
- Частота данных: каждые 15 минут
- Данные должны быть отсортированы по времени
- Допустимые значения: от 0 до 99% квантиля
- Формат файла: .xlsx
- Заголовки в первой строке: datetime, power_ac
Файл успешно загружен и очищен.
Ошибка: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.
Ошибка: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.
Обучение ARIMA (665 точек за 14 дней)...


C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\a

Слишком мало точек после очистки от NaN.
Обучение ARIMA (665 точек за 14 дней)...


C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\a

Слишком мало точек после очистки от NaN.
Файл успешно загружен и очищен.
Обучение ARIMA (1320 точек за 14 дней)...


...

Слишком мало точек после очистки от NaN.
Обучение ARIMA (1320 точек за 14 дней)...


C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\User\a

Слишком мало точек после очистки от NaN.
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
MAE LSTM: 57.12 кВт (5.1%)
MILP завершена. MAE: 0.00 кВт (0.0%)
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
MAE LSTM: 51.44 кВт (4.7%)


SystemExit: 0

C:\Users\User\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
